# Supplemental Figure 3, Panel E

Heatmaps with ranked NES from GSEA, along with barcode plots showing locations of "Stimulus/Response" GO:BP terms (from BAFinh vs Veh differential expression at 1h, 6h, and 30h). Violin plots show NES distribution for stimulus and non-stimulus terms, with P-values from wilcoxon rank-sum tests

*Output: PanelE.pdf*

# Setup

In [1]:
library(tidyverse)
library(fgsea)
library(msigdbr)
library(patchwork)
library(RColorBrewer)
library(rtracklayer)
library(ggrastr)
library(here)
here::i_am("figures/SuppFig3/PanelD/PanelD.ipynb")
theme_set(theme_bw() + theme(text=element_text(size=15), panel.grid=element_blank()))

#function to perform GSEA analysis and annotate results based on whether GO:BP term is of type STIMULUS or RESPONSE
gsea <- function(table, genesets){
    set.seed(1)
    ranking.vals <- table$log2FoldChange
    names(ranking.vals) <- table$gene

    res <- fgsea(pathways=genesets, stats=ranking.vals,minSize=10, )
    res$isStim <- grepl("STIMULUS|RESPONSE", res$pathway)
    return(res)
}

collapse_deg_duplicates <- function(df,
                                    gene_col = "gene",
                                    padj_col = "padj",
                                    lfc_col  = "log2FoldChange") {
  df %>%
    filter(!is.na(.data[[gene_col]])) %>%
    group_by(.data[[gene_col]]) %>%
    arrange(.data[[padj_col]], desc(abs(.data[[lfc_col]]))) %>%
    dplyr::slice(1) %>%
    ungroup()
}

#plot that takes GSEA results as inputs and returns a heatmap of ranked NES values as well as an annotation bar
#denoting whether a given term is of type STIMULUS/RESPONSE
#wilcoxon rank-sum test performed to test whether NES values for STIMULUS/RESPONSE terms are more negative than other terms
# includes violin plots showing NES distribution for each GO term set, with quantiles (25th, median, 75th) indicated 

hm_plot_violin <- function(gsea_table){
    gsea_table <- na.omit(gsea_table) %>% mutate(negNES=NES<0)
    
    #statistical test for difference in distribution
    wilcox_p <- wilcox.test(gsea_table[gsea_table$isStim==TRUE,]$NES, gsea_table[gsea_table$isStim==FALSE,]$NES, alternative = "less")$p.value

    #identify position of median term (by NES) in stim and non-stim GO term sets
    pos <- gsea_table %>%
        arrange(NES) %>%
        mutate(rank = row_number()) %>%
        group_by(isStim) %>%
        summarize(
        med_NES  = median(NES),
        med_rank = rank[which.min(abs(NES - median(NES)))],
        .groups="drop"
      )
    
    ranked_hm <- gsea_table %>% 
        ggplot() + 
        rasterize(geom_tile(aes(x='',y=reorder(pathway, NES), fill=NES)), dpi=300) + 
        theme(axis.text.y=element_blank(), rect=element_blank()) +
        #scale_fill_gradient2(low=brewer.pal(3,"Paired")[2], mid="white", high=brewer.pal(6,"Paired")[6], limits=c(-max_NES,max_NES))
        scale_fill_gradient2(low="darkblue", mid="white", high="darkred", limits=c(-2.5,2.5)) +
        ylab(paste("GO:BP terms, ranked by NES\nn = ", nrow(gsea_table))) +
        ggtitle(paste0("p_wilcox=", format(wilcox_p, digits=3)))


    annot <- gsea_table %>% 
                ggplot() + 
                rasterize(geom_tile(aes(x='',y=reorder(pathway, NES), fill=isStim)), dpi=300) + 
                theme(axis.text.y=element_blank(), rect=element_blank()) +
                scale_fill_manual(values=c("white", "black")) +
                scale_color_manual(values=c("grey", "black"))

    annot2 <- gsea_table %>% 
                ggplot(aes(x=isStim,y=NES)) + geom_violin(draw_quantiles=seq(0.25,1,by=.25))
    
    comb.plot <- (ranked_hm + annot + annot2) + plot_layout(widths=c(1,.2, 1), guides="collect")
    
   
    return(comb.plot)
}

sessionInfo()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:b

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] here_1.0.2           ggrastr_1.0.2        rtracklayer_1.58.0  
 [4] GenomicRanges_1.50.2 GenomeInfoDb_1.34.9  IRanges_2.32.0      
 [7] S4Vectors_0.36.2     BiocGenerics_0.44.0  RColorBrewer_1.1-3  
[10] patchwork_1.3.0      msigdbr_7.5.1        fgsea_1.24.0        
[13] lub

# load GO:BP terms from msigdb

In [3]:
#load GO:BP terms from msigdbr
all_gene_sets = msigdbr(species = "Homo sapiens")
keeper.cats <- c("GO:BP")
all_gene_sets <- all_gene_sets[which(all_gene_sets$gs_subcat %in% keeper.cats),]

#convert terms to fgsea_compatible format
msigdbr_list = split(x = all_gene_sets$gene_symbol, f = all_gene_sets$gs_name)

# Load data

In [4]:
#load DEG datasets
brm_1h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878", "BAFinh_stimulus",
    "BRM_1h_DEGs.tsv"
  ),
  sep = "\t"
)

brm_6h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878", "BAFinh_stimulus",
    "BRM_6h_DEGs.tsv"
  ),
  sep = "\t"
)

brm_30h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878",
    "BRM_30h_DEGs.tsv"
  ),
  sep = "\t"
)


# Run GSEA

In [5]:
#run GSEA
brm_1h_gsea <- gsea(brm_1h,genesets=msigdbr_list)
brm_6h_gsea <- gsea(brm_6h, genesets=msigdbr_list)
brm_30h_gsea <- gsea(brm_30h, genesets=msigdbr_list)

Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are duplicate gene names, fgsea may produce unexpected results.”
Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are duplicate gene names, fgsea may produce unexpected results.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 7 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.”
Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, g

# Generate plots

In [6]:
p_1h <- hm_plot_violin(brm_1h_gsea)
p_6h <- hm_plot_violin(brm_6h_gsea)
p_30h <- hm_plot_violin(brm_30h_gsea)

(p_1h | p_6h | p_30h) + plot_layout(guides="collect")
ggsave("PanelE.pdf", width=16)

Saving 16 x 7 in image
